In [1]:
cd ..

c:\Users\Learning\Project\ThesisProd


In [2]:
# Import necessary libraries
import nltk
nltk.download('punkt')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score_func
import torch

# For BARTScore and GPTScore
from transformers import BartForConditionalGeneration, BartTokenizer
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# For QSTS and QRelScore
from sentence_transformers import SentenceTransformer, util
# from evalpackage.qrelscore import QRelScore

# # Define the context sentence and generated MCQ question
# context_sentence = "Machine learning is a field of artificial intelligence that uses algorithms to learn from and make predictions on data."
# generated_mcq_question = "What is machine learning?"
# options = [
#     "A field of artificial intelligence that uses algorithms",
#     "A form of supervised learning only",
#     "A programming language",
#     "A method of organizing data"
# ]
# # context_sentence = "A high-degree polynomial kernel introduces high complexity to the decision boundary, which can lead to overfitting, especially when the training dataset is small. The SVM model may fit the noise in the training data rather than capturing the underlying pattern, reducing generalization to unseen data."
# # generated_mcq_question = "In the context of Support Vector Machines (SVM), which of the following scenarios is most likely to lead to overfitting when classifying a dataset?"
# # options = [
# #     "Choosing a linear kernel for a dataset that is linearly separable.",
# #     "Using a high-degree polynomial kernel on a small training dataset.",
# #     "Selecting a Gaussian (RBF) kernel with a large value for the hyperparameter 𝛾"
# #     "Setting the regularization parameter C to a very small value."
# # ]

# mcq_text = generated_mcq_question + " " + " ".join(options)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
c:\Users\PC\anaconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
import sys

import gradio as gr

from dotenv import load_dotenv
from core.retriever.Retriever import Retriever
from core.ingestion.preprocessing.storage.FaissStore import FaissStore
from core.llm.TeacherLLM import TeacherBot
from core.llm.AssistantLLM import AssistantBot

# Load environment variables from a .env file
load_dotenv()
# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

Custom LLM initialized with model: gpt-3.5-turbo


In [4]:
# path = "data/using/"
# faiss_store = FaissStore(documents_path = path)
# nodes = faiss_store.get_nodes()

In [5]:
# store = Retriever(nodes=nodes)
# retriever = store.get_retriever(top_k = 7)
# context = store.get_big_context(retriever = retriever, query = "Machine Learning")

In [6]:
context = 'Machine learning is a revolutionary concept that allows computers to learn how to perform tasks without explicit programming. Coined in 1959 by Arthur Samuel at IBM, machine learning involves feeding data into an algorithm to improve outcomes through experience, akin to organic learning. Today, predictive models are ubiquitous in our daily lives, serving to classify data and make predictions about future outcomes. The process begins with acquiring and cleaning vast amounts of data, as the quality of data directly impacts the results. Data scientists engage in feature engineering to transform raw data into meaningful features that represent the problem accurately.\n\nThe data is then split into training and testing sets, with the training data used to build a model and the testing data to validate its accuracy. Choosing an algorithm is a crucial step, ranging from simple statistical models like linear regression to complex neural networks that automatically create features. These algorithms improve by comparing their predictions to an error function, such as accuracy for classification problems or mean absolute error for regression problems.\n\nPython is the preferred language for data scientists, although R and Julia are also popular choices. Various frameworks support the machine learning process, making it more accessible. The end result of the process is a model, a file that takes input data and produces predictions to minimize the error it was optimized for. These models can be embedded in devices or deployed to the cloud to create real-world products.\n\nMachine learning has transformed industries and continues to evolve rapidly, offering endless possibilities for innovation and problem-solving.'

In [7]:
llm = TeacherBot()
respond = llm.create_question(context = context, ques_type = "Apply")

Custom LLM initialized with model: gpt-3.5-turbo


In [8]:
respond

[['Question: What is the primary purpose of feature engineering in machine learning?',
  'A. To acquire data',
  'B. To clean data',
  'C. To transform raw data into meaningful features',
  'D. To split data into training and testing sets',
  'Answer: C',
  'Context: Machine learning is a revolutionary concept that allows computers to learn how to perform tasks without explicit programming.'],
 ['Question: Why is choosing the right algorithm crucial in the machine learning process?',
  'A. To split data into training and testing sets',
  'B. To validate the accuracy of the model',
  'C. To transform raw data into meaningful features',
  'D. To improve predictions by comparing to an error function',
  'Answer: B',
  'Context: The data is then split into training and testing sets, with the training data used to build a model and the testing data to validate its accuracy.'],
 ['Question: What is the preferred language for data scientists in the field of machine learning?',
  'A. R',
  'B.

In [9]:
def compute_bleu4(context: str = None, mcq_text: str = None) -> float:
    """
    Compute BLEU-4 score between the context and generated MCQ question.
    """
    # Tokenize the context and generated MCQ question
    reference = nltk.word_tokenize(context.lower())
    candidate = nltk.word_tokenize(mcq_text.lower())
    
    # Calculate BLEU-4 score
    bleu_score = sentence_bleu([reference], candidate, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=SmoothingFunction().method1)
    
    return bleu_score

In [10]:
def compute_rougeL(context: str = None, mcq_text: str = None) -> float:
    """
    Compute ROUGE-L score between the context and generated MCQ question.
    """
    # Initialize ROUGE scorer
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    
    # Calculate ROUGE-L score
    scores = scorer.score(context, mcq_text)
    
    return scores['rougeL'].fmeasure

In [11]:
def compute_qrel(context: str = None, mcq_text: str = None):
    """
    Compute QSTS score between the context and generated MCQ question.
    """
    # Load the QSTS model
    qrel_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    
    # Encode the context and generated MCQ question
    context_embedding = qrel_model.encode(context, convert_to_tensor=True)
    mcq_text_embedding = qrel_model.encode(mcq_text, convert_to_tensor=True)
    
    # Calculate cosine similarity
    cosine_similarity = util.pytorch_cos_sim(context_embedding, mcq_text_embedding)
    
    return cosine_similarity.item()

In [13]:
for i in range(len(respond)):
    context = respond[i][6]
    mcq_text = respond[i][0] + " " + " ".join(respond[i][1:5])
    print(f'Question {i+1}:' + "\n")
    print(f"BLEU-4 score: {compute_bleu4(context, mcq_text):.4f}")
    print(f"ROUGE-L Score: {compute_rougeL(context, mcq_text):.4f}")
    print(f"QRelScore: {compute_qrel(context, mcq_text):.4f}")
    print("------------------------------------------------------------")

Question 1:

BLEU-4 score: 0.0132
ROUGE-L Score: 0.1786
QRelScore: 0.4937
------------------------------------------------------------
Question 2:

BLEU-4 score: 0.0994
ROUGE-L Score: 0.2895
QRelScore: 0.4493
------------------------------------------------------------
Question 3:

BLEU-4 score: 0.2568
ROUGE-L Score: 0.4500
QRelScore: 0.7980
------------------------------------------------------------
Question 4:

BLEU-4 score: 0.1881
ROUGE-L Score: 0.4179
QRelScore: 0.3871
------------------------------------------------------------
Question 5:

BLEU-4 score: 0.1001
ROUGE-L Score: 0.2462
QRelScore: 0.5277
------------------------------------------------------------
